<a href="https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected.")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Rebuild Week 5 data
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        AVG(gsc_impressions) as avg_impressions,
        AVG(gsc_clicks) as avg_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

april = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) as april_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

data = march.merge(april, on=["client_hash_id", "content_hash_id"], how="inner")

# ORIGINAL (leaky) label from Week 5 — kept here to show the before/after
data["is_declining_leaky"] = (data["april_clicks"] < data["avg_clicks"] * 30 * 0.85).astype(int)

# FIXED label — compare April's rate to March's rate using a ratio,
# and drop avg_clicks from the feature set instead, so no feature is
# arithmetically embedded in the label
data["march_daily_rate"] = data["avg_clicks"]
data["april_daily_rate"] = data["april_clicks"] / 30
data["is_declining"] = (data["april_daily_rate"] < data["march_daily_rate"] * 0.85).astype(int)

# Fixed feature set: drop avg_clicks (the leaky one), keep the rest
feature_cols_fixed = ["avg_impressions", "avg_position", "ctr"]
feature_cols_original = ["avg_impressions", "avg_clicks", "avg_position", "ctr"]

X_fixed = data[feature_cols_fixed].fillna(0)
X_original = data[feature_cols_original].fillna(0)
y = data["is_declining"]
groups = data["client_hash_id"]

print(f"Total pages: {len(data)}")
print(f"Declining rate (fixed label): {y.mean():.3f}")

Connected.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages: 158549
Declining rate (fixed label): 0.280


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## Paper Findings + My Methodology Questions

**Finding 1: The Freshness Multiplier (Finding #4)**
The paper reports that pages refreshed in the last 30 days show a
growth-to-decline ratio as high as 5.43:1 in the 31-90 day freshness
window, and a separate 365+ refreshed cohort shows a 1.6x health lift
and 52x impression lift versus stale pages.

**My methodology question:** How is "growth" defined relative to the
refresh event itself — is the growth window measured strictly *after*
the refresh date, with a clean gap from the freshness signal used to
select refresh candidates? If a page is refreshed because it already
showed early signs of movement, the refresh and the growth could be
correlated for reasons other than the refresh itself.

**Finding 2: 30-Day Momentum Model (Part IV)**
The paper reports a model predicting 30-day improvement with 95% accuracy
on same-brand unseen pages, with "Prior 30d Impressions" as the top predictor.

**My methodology question:** Is "prior 30-day impressions" measured from a
strictly earlier window than the outcome, with no overlap day shared
between feature and label windows? I ask this directly because I found
this exact problem in my own Week-5 model (see Section 2) — a feature
that was arithmetically embedded in my label definition produced an
unrealistic, near-perfect score.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Before/After: Fixing My Own Leakage

In Week 5, my label was defined as `april_clicks < avg_clicks * 30 * 0.85`
— and `avg_clicks` was also one of my model's input features. This created
a mechanical, formulaic link between a feature and the label, producing an
unrealistic Random Forest Precision@50 of 1.000 and zero false positives
in the top 50 — a clear leakage signal, not genuine predictive skill.

**The fix:** I removed `avg_clicks` from the feature set (keeping only
avg_impressions, avg_position, and ctr) and redefined the label using an
explicit daily-rate comparison, so no single feature is arithmetically
embedded in the label formula.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Client-grouped split, using the FIXED features and label
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_fixed, y, groups=groups))

X_train, X_test = X_fixed.iloc[train_idx], X_fixed.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

logreg_fixed = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_fixed.fit(X_train, y_train)
fixed_score = logreg_fixed.predict_proba(X_test)[:, 1]
fixed_p50 = precision_at_k(fixed_score, y_test.values, 50)

# For comparison: the ORIGINAL leaky setup (Week 5), same split indices
X_train_leaky, X_test_leaky = X_original.iloc[train_idx], X_original.iloc[test_idx]
y_leaky = data["is_declining_leaky"]
y_train_leaky, y_test_leaky = y_leaky.iloc[train_idx], y_leaky.iloc[test_idx]

logreg_leaky = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_leaky.fit(X_train_leaky, y_train_leaky)
leaky_score = logreg_leaky.predict_proba(X_test_leaky)[:, 1]
leaky_p50 = precision_at_k(leaky_score, y_test_leaky.values, 50)

print("=== Before/After: Leakage Fix ===")
print(f"BEFORE (Week 5, avg_clicks in features + label): Precision@50 = {leaky_p50:.3f}")
print(f"AFTER (avg_clicks removed, clean label):          Precision@50 = {fixed_p50:.3f}")
print(f"Gap: {leaky_p50 - fixed_p50:.3f}")

=== Before/After: Leakage Fix ===
BEFORE (Week 5, avg_clicks in features + label): Precision@50 = 0.940
AFTER (avg_clicks removed, clean label):          Precision@50 = 0.840
Gap: 0.100


[Fill in with real numbers, e.g.: "Removing the leaky feature dropped
Precision@50 from X.XXX to X.XXX — that gap is the size of the artificial
signal my original label formula created. The lower, honest number is the
one I'll carry forward into the paper."]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Feature correlation with label (leakage smell test) ===")
for col in feature_cols_fixed:
    corr = data[col].corr(data["is_declining"])
    print(f"{col}: correlation with label = {corr:.3f}")

print("\nFixed feature columns:", feature_cols_fixed)
print("avg_clicks removed from features?", "avg_clicks" not in feature_cols_fixed)

=== Feature correlation with label (leakage smell test) ===
avg_impressions: correlation with label = 0.169
avg_position: correlation with label = -0.183
ctr: correlation with label = 0.180

Fixed feature columns: ['avg_impressions', 'avg_position', 'ctr']
avg_clicks removed from features? True


## Leakage Audit

After the fix, none of my three features (avg_impressions, avg_position,
ctr) are arithmetically embedded in the label definition — the label now
compares April's actual daily rate to March's, using a ratio threshold,
with no shared feature. This is the corrected version of the leakage I
found and explained in Section 2.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## Claim Rewrite — Safe Language

**Before (overclaiming, and based on a leaky result):** "My Random Forest
model perfectly predicts which pages will decline (Precision@50 = 1.000)."

**After (safe, honest):** "After removing a feature that was arithmetically
linked to my label definition, my corrected model's ranked output is
observed to [beat/roughly match/underperform] the Week-4 baseline under a
client-grouped split. This is a directional, decision-support signal, not
a guarantee about any individual page, and the earlier near-perfect score
was an artifact of label leakage, not genuine predictive skill."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.